# <span style="color: #3498db;">Checkers with Minimax Algorithm</span>

In this assignment, your tasks are as follows:

- Use a **pre-built Checkers game engine** (board, pieces, valid-move generation) and complete the TODOS we have put.
- Implement the **Minimax search algorithm**.
- Extend Minimax with **Alpha–Beta pruning**.
- And also, experiment with the **evaluation function** to fine tune the AI's playing style to make it as good as possible. (different heuristics)

In [13]:
import time
import pygame
import matplotlib.pyplot as plt

from copy import deepcopy

RED = (255,0,0)
WHITE = (255, 255, 255)

FPS = 60

In [14]:
import sys
!{sys.executable} -m pip install --user --break-system-packages --no-cache-dir -i https://mirror-pypi.runflare.com/simple pygame

Looking in indexes: https://mirror-pypi.runflare.com/simple


# <span style="color: #126ca8; font-size:32px;">Actual Game Implementation</span>

In [15]:
WIDTH, HEIGHT = 800, 800
ROWS, COLS = 8, 8
SQUARE_SIZE = WIDTH//COLS

RED = (255, 0, 0)
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
BLUE = (0, 0, 255)
GREY = (128,128,128)

CROWN = pygame.transform.scale(pygame.image.load('assets/crown.png'), (44, 25))

FPS = 60

WIN = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption('Checkers')

In [16]:
def get_row_col_from_mouse(pos):
    x, y = pos
    row = y // SQUARE_SIZE
    col = x // SQUARE_SIZE
    return row, col

In [17]:
class Piece:
    PADDING = 15
    OUTLINE = 2

    def __init__(self, row, col, color):
        self.row = row
        self.col = col
        self.color = color
        self.king = False
        self.x = 0
        self.y = 0
        self.calc_pos()

    def calc_pos(self):
        self.x = SQUARE_SIZE * self.col + SQUARE_SIZE // 2
        self.y = SQUARE_SIZE * self.row + SQUARE_SIZE // 2

    def make_king(self):
        self.king = True

    def draw(self, win):
        radius = SQUARE_SIZE//2 - self.PADDING
        pygame.draw.circle(win, GREY, (self.x, self.y), radius + self.OUTLINE)
        pygame.draw.circle(win, self.color, (self.x, self.y), radius)
        if self.king:
            win.blit(CROWN, (self.x - CROWN.get_width()//2, self.y - CROWN.get_height()//2))

    def move(self, row, col):
        self.row = row
        self.col = col
        self.calc_pos()

    def __repr__(self):
        return str(self.color)

In [18]:
class Board:
    def __init__(self):
        self.board = []
        self.red_left = self.white_left = 12
        self.red_kings = self.white_kings = 0
        self.create_board()

    def draw_squares(self, win):
        win.fill(BLACK)
        for row in range(ROWS):
            for col in range(row % 2, COLS, 2):
                pygame.draw.rect(win, RED, (row*SQUARE_SIZE, col *SQUARE_SIZE, SQUARE_SIZE, SQUARE_SIZE))

    def evaluate(self):
     
        white_score = self.white_left + (self.white_kings * 1.5)
        red_score = self.red_left + (self.red_kings * 1.5)
        
        white_adv = 0
        red_adv = 0
        
        for piece in self.get_all_pieces((255, 255, 255)): 
            if not piece.king:
                white_adv += piece.row
        
        for piece in self.get_all_pieces((255, 0, 0)): 
            if not piece.king:
                red_adv += (7 - piece.row)
        
        white_safe = 0
        red_safe = 0
        
        for piece in self.get_all_pieces((255, 255, 255)): 
            if piece.col == 0 or piece.col == 7:
                white_safe += 1
        
        for piece in self.get_all_pieces((255, 0, 0)): 
            if piece.col == 0 or piece.col == 7:
                red_safe += 1
        
        evaluation = (white_score - red_score) * 10
        evaluation += (white_adv - red_adv) * 2
        evaluation += (white_safe - red_safe) * 3
        
        return evaluation


    def get_all_pieces(self, color):
        pieces = []
        for row in self.board:
            for piece in row:
                if piece != 0 and piece.color == color:
                    pieces.append(piece)
        return pieces

    def move(self, piece, row, col):
        self.board[piece.row][piece.col], self.board[row][col] = self.board[row][col], self.board[piece.row][piece.col]
        piece.move(row, col)

        if row == ROWS - 1 or row == 0:
            piece.make_king()
            if piece.color == WHITE:
                self.white_kings += 1
            else:
                self.red_kings += 1 

    def get_piece(self, row, col):
        return self.board[row][col]

    def create_board(self):
        for row in range(ROWS):
            self.board.append([])
            for col in range(COLS):
                if col % 2 == ((row +  1) % 2):
                    if row < 3:
                        self.board[row].append(Piece(row, col, WHITE))
                    elif row > 4:
                        self.board[row].append(Piece(row, col, RED))
                    else:
                        self.board[row].append(0)
                else:
                    self.board[row].append(0)

    def draw(self, win):
        self.draw_squares(win)
        for row in range(ROWS):
            for col in range(COLS):
                piece = self.board[row][col]
                if piece != 0:
                    piece.draw(win)

    def remove(self, pieces):
        for piece in pieces:
            if piece:
                self.board[piece.row][piece.col] = 0

                if piece.color == RED:
                    self.red_left -= 1
                    if piece.king:
                        self.red_kings -= 1
                else:
                    self.white_left -= 1
                    if piece.king:
                        self.white_kings -= 1

    def winner(self):
        if self.red_left <= 0:
            return WHITE
        if self.white_left <= 0:
            return RED
        return None

    def get_valid_moves(self, piece):
        moves = {}
        left = piece.col - 1
        right = piece.col + 1
        row = piece.row

        if piece.color == RED or piece.king:
            moves.update(self._traverse_left(row -1, max(row-3, -1), -1, piece.color, left))
            moves.update(self._traverse_right(row -1, max(row-3, -1), -1, piece.color, right))
        if piece.color == WHITE or piece.king:
            moves.update(self._traverse_left(row +1, min(row+3, ROWS), 1, piece.color, left))
            moves.update(self._traverse_right(row +1, min(row+3, ROWS), 1, piece.color, right))

        return moves

    def _traverse_left(self, start, stop, step, color, left, skipped=[]):
        moves = {}
        last = []
        for r in range(start, stop, step):
            if left < 0:
                break

            current = self.board[r][left]
            if current == 0:
                if skipped and not last:
                    break
                elif skipped:
                    moves[(r, left)] = last + skipped
                else:
                    moves[(r, left)] = last

                if last:
                    if step == -1:
                        row = max(r-3, 0)
                    else:
                        row = min(r+3, ROWS)
                    moves.update(self._traverse_left(r+step, row, step, color, left-1,skipped=last))
                    moves.update(self._traverse_right(r+step, row, step, color, left+1,skipped=last))
                break
            elif current.color == color:
                break
            else:
                last = [current]

            left -= 1

        return moves

    def _traverse_right(self, start, stop, step, color, right, skipped=[]):
        moves = {}
        last = []
        for r in range(start, stop, step):
            if right >= COLS:
                break

            current = self.board[r][right]
            if current == 0:
                if skipped and not last:
                    break
                elif skipped:
                    moves[(r,right)] = last + skipped
                else:
                    moves[(r, right)] = last

                if last:
                    if step == -1:
                        row = max(r-3, 0)
                    else:
                        row = min(r+3, ROWS)
                    moves.update(self._traverse_left(r+step, row, step, color, right-1,skipped=last))
                    moves.update(self._traverse_right(r+step, row, step, color, right+1,skipped=last))
                break
            elif current.color == color:
                break
            else:
                last = [current]

            right += 1

        return moves

In [19]:
class Game:
    def __init__(self, win):
        self._init()
        self.win = win

    def update(self):
        self.board.draw(self.win)
        self.draw_valid_moves(self.valid_moves)
        pygame.display.update()

    def _init(self):
        self.selected = None
        self.board = Board()
        self.turn = RED
        self.valid_moves = {}

    def winner(self):
        piece_winner = self.board.winner()
        if piece_winner is not None:
            return piece_winner

        moves = 0
        for p in self.board.get_all_pieces(self.turn):
            moves += len(self.board.get_valid_moves(p))

        if moves == 0:
            return WHITE if self.turn == RED else RED

        return None

    def reset(self):
        self._init()

    def select(self, row, col):
        if self.selected:
            result = self._move(row, col)
            if not result:
                self.selected = None
                self.select(row, col)

        piece = self.board.get_piece(row, col)
        if piece != 0 and piece.color == self.turn:
            self.selected = piece
            self.valid_moves = self.board.get_valid_moves(piece)
            return True

        return False

    def _move(self, row, col):
        piece = self.board.get_piece(row, col)
        if self.selected and piece == 0 and (row, col) in self.valid_moves:
            self.board.move(self.selected, row, col)
            skipped = self.valid_moves[(row, col)]
            if skipped:
                self.board.remove(skipped)
            self.change_turn()
        else:
            return False

        return True

    def draw_valid_moves(self, moves):
        for move in moves:
            row, col = move
            pygame.draw.circle(self.win, BLUE, (col * SQUARE_SIZE + SQUARE_SIZE//2, row * SQUARE_SIZE + SQUARE_SIZE//2), 15)

    def change_turn(self):
        self.valid_moves = {}
        if self.turn == RED:
            self.turn = WHITE
        else:
            self.turn = RED

    def get_board(self):
        return self.board

    def ai_move(self, board):
        self.board = board
        self.change_turn()

# <span style="color: #126ca8; font-size:32px;">Minimax AI Implementation</span>

In [20]:
def simulate_move(piece, move, board, skip):
   
    board.move(piece, move[0], move[1])
    if skip:
        board.remove(skip)

    return board


def get_all_moves(board, color):
    
    moves = []

    for piece in board.get_all_pieces(color):
        valid_moves = board.get_valid_moves(piece)
        for move, skip in valid_moves.items():
            temp_board = deepcopy(board)
            temp_piece = temp_board.get_piece(piece.row, piece.col)
            new_board = simulate_move(temp_piece, move, temp_board, skip)
            moves.append(new_board)

    return moves

# <span style="color: #126ca8; font-size:32px;">Base (No Alpha/Beta Pruning)</span>

In [21]:
def minimax(position, depth, max_player: bool):
    if depth == 0 or position.winner() is not None:
        return position.evaluate(), position

    if max_player: 
        max_eval = float('-inf')
        best_move = None
        for move in get_all_moves(position, WHITE):
            evaluation = minimax(move, depth-1, False)[0]
            if evaluation > max_eval:
                max_eval = evaluation
                best_move = move
        return max_eval, best_move
        
    else: 
        min_eval = float('inf')
        best_move = None
        for move in get_all_moves(position, RED):
            evaluation = minimax(move, depth-1, True)[0]
            if evaluation < min_eval:
                min_eval = evaluation
                best_move = move
        return min_eval, best_move


# <span style="color: #126ca8; font-size:32px;">Optimized Version (With Alpha/Beta Pruning)</span>

In [22]:
def minimax_alpha_beta(position, depth, alpha, beta, max_player: bool):
    if depth == 0 or position.winner() is not None:
        return position.evaluate(), position

    if max_player: 
        max_eval = float('-inf')
        best_move = None
        for move in get_all_moves(position, WHITE):
            evaluation = minimax_alpha_beta(move, depth-1, alpha, beta, False)[0]
            if evaluation > max_eval:
                max_eval = evaluation
                best_move = move
            
            alpha = max(alpha, evaluation)
            if beta <= alpha: 
                break
        return max_eval, best_move
        
    else: 
        min_eval = float('inf')
        best_move = None
        for move in get_all_moves(position, RED):
            evaluation = minimax_alpha_beta(move, depth-1, alpha, beta, True)[0]
            if evaluation < min_eval:
                min_eval = evaluation
                best_move = move
                
            beta = min(beta, evaluation)
            if beta <= alpha: 
                break
        return min_eval, best_move


In [23]:
def play_ai_vs_ai(search_depth=4, use_pruning=True, track_nodes=False):

    board = Board()
    turn = RED  
    moves_count = 0

    if track_nodes:
        global NODES_SEEN
        NODES_SEEN = 0

    start_time = time.time()

    while True:
        winner = board.winner()
        if winner is not None:
            elapsed = time.time() - start_time
            return winner, moves_count, elapsed

        all_moves = get_all_moves(board, turn)
        if not all_moves:
            elapsed = time.time() - start_time
            winner = WHITE if turn == RED else RED
            return winner, moves_count, elapsed

        if use_pruning:
            _, best_board = minimax_alpha_beta(
                position=board,
                depth=search_depth,
                alpha=float('-inf'),
                beta=float('inf'),
                max_player=(turn == WHITE)
            )
        else:
            _, best_board = minimax(
                position=board,
                depth=search_depth,
                max_player=(turn == WHITE)
            )

        if best_board is None:
            best_board = all_moves[0]

        board = best_board
        moves_count += 1

        turn = WHITE if turn == RED else RED


def print_checkers_results(results, depth, pruning, total_time, avg_moves):
    pruning_status = "Enabled" if pruning else "Disabled"
    print("=" * 60)
    print(f"Results for Depth={depth} | Alpha-Beta: {pruning_status}")
    print("=" * 60)
    
    white_wins = results.count((255, 255, 255))
    red_wins = results.count((255, 0, 0))
    draws = results.count(None)
    
    print(f"Games played: {len(results)}")
    print(f"White wins: {white_wins} ({(100*white_wins/len(results)) if len(results) else 0:.1f}%)")
    print(f"Red wins: {red_wins} ({(100*red_wins/len(results)) if len(results) else 0:.1f}%)")
    print(f"Draws/Stuck: {draws} ({(100*draws/len(results)) if len(results) else 0:.1f}%)")
    print(f"Total time: {total_time:.2f}s")
    print(f"Avg moves: {avg_moves:.1f}")
    print(f"Avg time/game: {(total_time/len(results)) if len(results) else 0:.2f}s")
    print("=" * 60 + "\n")

def check_checkers_results(max_depth=3, games_per_setting=2):
    for pruning in [True, False]:
        print(f"\nTesting {'WITH' if pruning else 'WITHOUT'} Alpha-Beta Pruning")
        for depth in range(1, max_depth + 1):
            results = []
            total_time = 0
            total_moves = 0
            
            print(f"  Testing depth {depth}...", end=" ")
            for _ in range(games_per_setting):
                winner, moves_count, elapsed = play_ai_vs_ai(search_depth=depth, use_pruning=pruning)
                results.append(winner)
                total_time += elapsed
                total_moves += moves_count
            print("Done!")
            
            avg_moves = total_moves / games_per_setting
            print_checkers_results(results, depth, pruning, total_time, avg_moves)

def run_analysis_for_plots(max_depth=4, games_per_setting=2):
    import matplotlib.pyplot as plt
    depths = list(range(1, max_depth + 1))
    times_plain, times_pruning = [], []
    
    print("Generating analysis plots...")
    for depth in depths:
        t_plain = sum(play_ai_vs_ai(depth, False)[2] for _ in range(games_per_setting))
        times_plain.append(t_plain / games_per_setting)
        
        t_pruning = sum(play_ai_vs_ai(depth, True)[2] for _ in range(games_per_setting))
        times_pruning.append(t_pruning / games_per_setting)
    
    plt.figure(figsize=(8, 5))
    plt.plot(depths, times_plain, 'ro-', label='Minimax (No Pruning)', linewidth=2)
    plt.plot(depths, times_pruning, 'go-', label='Minimax (Alpha-Beta)', linewidth=2)
    plt.xlabel('Search Depth')
    plt.ylabel('Average Time per Game (seconds)')
    plt.title('Execution Time Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nSpeedup from Alpha-Beta Pruning:")
    for i, depth in enumerate(depths):
        if times_pruning[i] > 0:
            speedup = times_plain[i] / times_pruning[i]
            print(f"  Depth {depth}: {speedup:.1f}x faster")


# <span style="color: #126ca8; font-size:32px;">Main Game Loop</span>

In [26]:
def main_game_loop():
    pygame.init() 
    WIN = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption('Checkers AI')
    run = True
    clock = pygame.time.Clock()
    game = Game(WIN)
    SEARCH_DEPTH = 4 
    while run:
        clock.tick(FPS)
        
        if game.turn == WHITE:
            _, new_board = minimax_alpha_beta(
                game.get_board(),
                depth=SEARCH_DEPTH,
                alpha=float('-inf'),
                beta=float('inf'),
                max_player=True
            )
            if new_board:
                game.ai_move(new_board)
        
        if game.winner() is not None:
            winner_color = "WHITE" if game.winner() == WHITE else "RED"
            print(f"Game Over! Winner: {winner_color}")
            run = False
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                run = False
            
            if event.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()
                row, col = get_row_col_from_mouse(pos)
                game.select(row, col)
        
        game.update()
    
    pygame.quit()

main_game_loop()
